# **Projekt DIGITS**
## 1. Cel: rozpoznanie cyfr 0-9 z nagrań

---
*Autorzy*:
- *Wojciech Salamak*
- *Amelia Sowa*
- *Szymon Stach*
- *Kamil Koterba*



## 2. Plan i architektura systemu

### 1. Architektura i przepływ danych

````mermaid
flowchart LR
    n1["Pliki WAV"] --> n2["Odczyt audio i metadanych z nazwy pliku"]
    n2 --> n3["Przygotowanie sygnału"]
    n3 --> n4["Ekstrakcja cech MFCC"]
    n4 --> n5["Trening"] & n6["Klasyfikacja"]

    n1@{ shape: rect}
    n2@{ shape: rounded}
    n3@{ shape: rounded}
    n4@{ shape: rounded}
    n5@{ shape: lin-proc}
    n6@{ shape: lin-proc}
    style n1 fill:transparent,stroke:#FF6D00
    style n2 fill:transparent
    style n3 fill:transparent
    style n4 fill:transparent
    style n5 fill:transparent,stroke:#00C853
    style n6 fill:transparent,stroke:#2962FF
````

Przyjęto architekturę generatywną. Sygnał audio jest opisywany za pomocą współczynników MFCC. Każda cyfra jest reprezentowana przez osobny model GMM. Metoda ta pozwoli na modelowanie rozkładu cech bez wymagania jednakowej długości nagrań.

System jest podzielony na dwie ścieżki: 
- Trening modeli na nagraniach z etykietami w nazwach plików,
- Klasyfikacja nagrania o nieznanej cyfrze

Obie ścieżki wykorzystują tą samą procedurę przygotowania sygnału i ekstrakcji cech.

````mermaid

flowchart LR
    n6["Powiązanie cech z etykietą"] --> n7["Połączenie ramek z tych samych cyfr"]
    n7 --> n8["Trening osobnego GMM cyfr 0-9"]
    n8 --> n9["Słownik modeli GMM"]
    n5["Trening"] --> n11["Cechy MFCC"]
    n11 --> n6
    n10["Klasyfikacja"] --> n13["Cechy MFCC"]
    n16["Wybór najwyższego wyniku"] --> n17["Rozpoznana cyfra 0-9"]
    n18@{ label: "<span style=\"background-color:\">Obliczanie log-podobieństwa przez każdy model</span><br>" } --> n16
    n13 --> n18
    n15["Słownik modeli GMM"] --> n18

    n6@{ shape: rounded}
    n7@{ shape: rounded}
    n8@{ shape: rounded}
    n9@{ shape: db}
    n5@{ shape: lin-proc}
    n11@{ shape: db}
    n10@{ shape: lin-proc}
    n13@{ shape: db}
    n16@{ shape: rounded}
    n18@{ shape: rounded}
    n15@{ shape: db}
    style n9 stroke:#FFD600
    style n5 fill:transparent,stroke:#00C853
    style n11 stroke:#FF6D00
    style n10 stroke:#2962FF
    style n13 stroke:#FF6D00
    style n17 stroke:#00C853
    style n15 stroke:#FFD600
````

### 2. Struktura danych przechowująca cechy MFCC
Cechy ekstraktowane z nagrań będą przechowywane jako lista. 
Każdy jej element odpowiada jednemu plikowi audio.


```python
recordings = [
    {
        "path": "training_data/JC1M1_4_.wav",
        "speaker": "JC",
        "digit": 4,
        "features": mfcc_matrix
    }
]
```
Pole `features` zawiera macierz Numpy o wymiarze: (liczba_ramek, liczba_wspolczynnikow_MFCC).
Każdy wiersz opisuje jedna ramkę czasową. 
<p>Osobna macierz dla każdego nagrania pozwoli przechowywać wypowiedzi rożnej długości.

### 3. Struktura danych przechowująca modele GMM
System wykorzysta dziesięć modeli GMM, jeden dla każdej cyfry. 
Wytrenowane modele będą przechowywane w słowniku `models`

```python
models = {
    0: gmm_0,
    1: gmm_1,
    2: gmm_2,
    # ...
    9: gmm_9,
}
```
Każdy model przechowuje wagi komponentów, ich wektory średnich i macierze kowariancje rozkładu cech danej cyfry. 
Słownik ten zwiąże modele z etykietami klas i umożliwi poźniejszą ocenę nagrania.


## 3. Przygotwanie danych

In [ ]:
# Pobieramy dane treningowe z google drive
from utils.drive import download_data
path = download_data("https://drive.google.com/drive/u/1/folders/1CF7Q7dWQSmmrpj92mDmCYXKQg-nz_Lm_", data_dir='training_data')

In [ ]:
from pathlib import Path
import re

data_dir = Path("training_data")
wav_paths = sorted(
    path for path in data_dir.rglob("*")
    if path.is_file() and path.suffix.lower() == ".wav"
)

recordings = []

for path in wav_paths: 
    match = re.search(r"_([0-9])_", path.stem)

    recordings.append({
        "path": path,
        "speaker": path.name[:2],
        "digit": int(match.group(1)),
    })

## 4. Parametryzacja

## 5. Model klasyfikacyjny


## 6. Walidacja

## 7. Ewaluacja

## 8. Wnioski